# bioseq bench `report.json` viewer

Self-contained, single-file notebook for inspecting a `report.json`
produced by `./build.sh bench`. The bench writes the file to
`build/report.json` (or whatever path was passed via the
`BIOSEQ_BENCH_REPORT_PATH` environment variable). Override the
`REPORT_PATH` constant in the next cell to point somewhere else.

## Reading order

Charts are rendered in 5 tiers, top-to-bottom:

1. **Headline** (`headline_*`) — the four charts that answer 'how does
   bioseq compare to minimap2?' at a glance. Two minimap2 baselines and
   the two bioseq KSW2 + scalar-mm variants whose chaining matches
   minimap2's. Read these first.
2. **Minimizer k/w sweep** (`kmer_window_*`) — accuracy and latency
   heatmaps showing how (k, w) parameter choices affect both
   minimap2 and our port. Sourced from `run_kw_sweep` in
   `matrix_runner.cppm`.
3. **Full algorithm grid** — the same accuracy / speed / Pareto views
   over the complete 14-algorithm matrix. Use to drill into any
   variant the headline charts skip.
4. **Latency distributions** — P50/P99, CDFs, box plots, tail
   amplification. Use when sizing latency budgets.
5. **Quality diagnostics** — position-offset histograms, MAPQ
   distribution and calibration, the per-algorithm summary table.

No external dependencies beyond `plotly` and the Python standard
library. To run:

1. Open in Jupyter (`jupyter notebook view_report.ipynb`) or VS Code.
2. Execute the cells top-to-bottom. The last cell renders every
   chart in the report grouped by the five tiers above.

Schema: `bioseq-bench-report-v1` — see
`main/bioseq/RESEARCH/BENCHMARKING_SPEEDUP/REPORT_PLOTLY_JSON_DATA.md`
for what each chart means.

In [1]:
!pip3 install plotly nbformat


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ----- configuration -----
# Either an absolute path or a path relative to the notebook directory.
# The bench writes to ./report.json by default; the build script invokes
# the binary from main/build/, so the canonical location is
# main/build/report.json.
REPORT_PATH = '../../../build/report.json'

import json
import os
import sys

import plotly.io
from IPython.display import display, Markdown

with open(REPORT_PATH, 'r') as report_file:
    report = json.load(report_file)

expected_schema = 'bioseq-bench-report-v1'
actual_schema = report.get('schema', '<missing>')
if actual_schema != expected_schema:
    print(
        f'WARNING: schema mismatch — expected {expected_schema!r}, '
        f'got {actual_schema!r}. The notebook may render incorrectly.',
        file=sys.stderr,
    )

print(f'Loaded {os.path.abspath(REPORT_PATH)}')
print(f'  schema:           {actual_schema}')
print(f'  sample_count:     {report.get("sample_count", "?")}')
print(f'  algorithm_count:  {report.get("algorithm_count", "?")}')
print(f'  chart_count:      {len(report.get("charts", {}))}')

Loaded /Users/styczynski/Code/bioseqdb/main/build/report.json
  schema:           bioseq-bench-report-v1
  sample_count:     1008
  algorithm_count:  14
  chart_count:      17


In [ ]:
# Render charts grouped by tier so the headline answer comes first
# and the deep-dive details follow. Each tier prints a markdown
# section header before its charts; charts within a tier render in
# the order they appear in the JSON (which is the order
# `chart_builder::emit_report` writes them).
#
# Tiers:
#   1. Headline: 'how does bioseq compare to minimap2?' answers (chart IDs
#      starting with `headline_`).
#   2. k/w sweep: how do minimizer length and window size affect quality
#      (chart IDs starting with `kmer_window_`).
#   3. Algorithm comparison: full 14-algorithm grid views.
#   4. Latency distributions: tail-shape charts for capacity planning.
#   5. Quality diagnostics: position offsets, MAPQ, summary table.
#
# Charts not assigned to a tier are listed at the end under 'Other'.

charts = report.get('charts', {})

# (tier_label, predicate) — first-match-wins, so order matters here.
tier_specs = [
    ('Tier 1 — Headline (bioseq vs. minimap2)',
     lambda cid: cid.startswith('headline_')),
    ('Tier 2 — Minimizer k/w sweep',
     lambda cid: cid.startswith('kmer_window_')),
    ('Tier 3 — Full algorithm grid',
     lambda cid: cid in {
        'seq_length_vs_speed_mean',
        'seq_length_vs_accuracy',
        'error_rate_vs_accuracy',
        'accuracy_heatmap_algorithm_vs_length',
        'accuracy_heatmap_algorithm_vs_length_error',
        'seq_length_vs_throughput',
        'speedup_vs_minimap2_reference_dp',
        'pareto_accuracy_vs_latency',
     }),
    ('Tier 4 — Latency distributions and tail behaviour',
     lambda cid: cid in {
        'seq_length_vs_speed_p99',
        'latency_distribution_box',
        'p99_over_mean_tail_amplification',
        'latency_cdf_per_algorithm',
        'seq_length_vs_memory',
     }),
    ('Tier 5 — Quality diagnostics',
     lambda cid: cid in {
        'position_offset_histogram',
        'mapq_distribution',
        'mapq_calibration',
        'algorithm_summary_table',
     }),
]

assigned = set()

for tier_label, predicate in tier_specs:
    matching_ids = [cid for cid in charts if cid not in assigned and predicate(cid)]
    if not matching_ids:
        continue
    display(Markdown(f'# {tier_label}'))
    for cid in matching_ids:
        chart = charts[cid]
        short_description = chart.get('short_description', cid)
        long_description = chart.get('long_description', '')
        plotly_payload = chart.get('plotly')
        if plotly_payload is None:
            display(Markdown(f'### `{cid}` — _no plotly payload, skipped_'))
            assigned.add(cid)
            continue
        display(Markdown(f'## `{cid}` — {short_description}'))
        try:
            figure = plotly.io.from_json(json.dumps(plotly_payload))
        except Exception as exc:  # noqa: BLE001
            display(Markdown(f'**plotly.io.from_json failed:** `{exc}`'))
            assigned.add(cid)
            continue
        figure.show()
        if long_description:
            display(Markdown(f'> {long_description}'))
        assigned.add(cid)

leftover_ids = [cid for cid in charts if cid not in assigned]
if leftover_ids:
    display(Markdown('# Other charts (not categorised)'))
    for cid in leftover_ids:
        chart = charts[cid]
        short_description = chart.get('short_description', cid)
        long_description = chart.get('long_description', '')
        plotly_payload = chart.get('plotly')
        if plotly_payload is None:
            display(Markdown(f'### `{cid}` — _no plotly payload, skipped_'))
            continue
        display(Markdown(f'## `{cid}` — {short_description}'))
        try:
            figure = plotly.io.from_json(json.dumps(plotly_payload))
        except Exception as exc:  # noqa: BLE001
            display(Markdown(f'**plotly.io.from_json failed:** `{exc}`'))
            continue
        figure.show()
        if long_description:
            display(Markdown(f'> {long_description}'))